In [1]:
import os
import requests
import docx
import logging
import random
import time
import psycopg2  # PostgreSQL integration
import streamlit as st
from bs4 import BeautifulSoup
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.chains import LLMChain, SequentialChain
from langchain_community.document_loaders import PyPDFLoader
import tiktoken
from dotenv import load_dotenv

In [2]:
# Load environment variables
load_dotenv(".env")
openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key:
    raise ValueError("OpenAI API key not found. Please check your .env file.")

In [3]:
# Initialize LLM Model
llm_model = "gpt-4o"
llm = ChatOpenAI(temperature=0.0, model=llm_model, openai_api_key=openai_api_key)

def count_tokens(text, model=llm_model):
    if not isinstance(text, str):
        return 0
    encoding = tiktoken.encoding_for_model(model)
    return len(encoding.encode(text))


In [4]:
# Define prompts
prompt_job_description = ChatPromptTemplate.from_template("""
    Extract key insights from job description:
    {job_description}
""")

prompt_profiler = ChatPromptTemplate.from_template("""
    Analyze resume and LinkedIn profile:
    Resume: {resume}
    LinkedIn Profile: {linkedin}
""")

prompt_cover_letter = ChatPromptTemplate.from_template("""
    Generate a personalized cover letter:
    Profile: {personal_profile}
    Job Description: {job_summary}
""")

prompt_proof_reader = ChatPromptTemplate.from_template("""
    Proofread and refine the cover letter:
    Draft: {cover_letter_draft}
""")


In [5]:
# Define LLM Chains
chain_one = LLMChain(llm=llm, prompt=prompt_job_description, output_key="job_summary")
chain_two = LLMChain(llm=llm, prompt=prompt_profiler, output_key="personal_profile")
chain_three = LLMChain(llm=llm, prompt=prompt_cover_letter, output_key="cover_letter_draft")
chain_four = LLMChain(llm=llm, prompt=prompt_proof_reader, output_key="cover_letter_final")

sequential_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=["resume", "linkedin", "job_description"],
    output_variables=["job_summary", "personal_profile", "cover_letter_draft", "cover_letter_final"],
    verbose=True
)


C:\Users\USER\AppData\Local\Temp\ipykernel_10180\3884750293.py:2: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  chain_one = LLMChain(llm=llm, prompt=prompt_job_description, output_key="job_summary")


In [6]:
# File Extraction Functions
def extract_text_from_pdf(file):
    temp_path = f"./temp_{file.name}"
    with open(temp_path, "wb") as f:
        f.write(file.getvalue())
    loader = PyPDFLoader(temp_path)
    text = "\n".join([page.page_content for page in loader.load()])
    os.remove(temp_path)
    return text

def extract_text_from_docx(file):
    doc = docx.Document(file)
    return "\n".join([para.text for para in doc.paragraphs])

In [7]:
# Job Description Fetching
def fetch_job_description(url):
    try:
        response = requests.get(url)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')
        return soup.get_text(strip=True)
    except requests.exceptions.RequestException as e:
        return None

In [8]:
# Cover Letter Generation
def cover_letter_gen(resume_file, linkedin_url, job_description_url, manual_job_desc, cover_letter_file):
    resume_content = extract_text_from_pdf(resume_file) if resume_file else ""
    job_description = fetch_job_description(job_description_url) or manual_job_desc.strip()
    linkedin_profile = linkedin_url or ""
    cover_letter_format = extract_text_from_docx(cover_letter_file) if cover_letter_file else ""
    
    inputs = {
        "resume": resume_content,
        "linkedin": linkedin_profile,
        "job_description": job_description
    }
    
    outputs = sequential_chain(inputs)
    return outputs.get("cover_letter_final", "Error generating cover letter")

In [9]:
# Database Storage
def save_to_db(name, email, cover_letter):
    try:
        conn = psycopg2.connect(
            dbname='your_db', user='your_user', password='your_password', host='your_host', port='your_port'
        )
        cursor = conn.cursor()
        cursor.execute("INSERT INTO cover_letters (name, email, content) VALUES (%s, %s, %s)", (name, email, cover_letter))
        conn.commit()
        cursor.close()
        conn.close()
    except Exception as e:
        logging.error(f"Database Error: {str(e)}")